# Network Analysis

Quick sanity checks and plots of a solved PyPSA-Earth network.

## Sync a solved network (from repo root)

```bash
rsync -avz --progress \
  engs2523@arc-login.arc.ox.ac.uk:/data/engs-df-green-ammonia/engs2523/pypsa-earth/results/<run-name>/networks/<network-file>.nc \
  results/<run-name>/networks/
```
Then update `RESULT_PATH` in the next cell to point at the downloaded file.

In [ ]:
# Download natura raster on first run
# import cartopy; from cartopy import feature as cf
# cf.NaturalEarthFeature("physical","land","10m")

In [ ]:
from pathlib import Path

RESULT_PATH_OVERRIDE = None

RUN_CANDIDATES = {
    # --- Base techs only (no H2 / NH3) ---
    "co2_limited": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2L-3h.nc"),
    "co2_zero": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2zero-3h.nc"),
    "co2_uncapped": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2nocap-3h.nc"),
    # --- With H2 (CCGT H2 + pipeline) ---
    "co2_limited_h2": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2L-3h-H2.nc"),
    "co2_zero_h2": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2zero-3h-H2.nc"),
    "co2_uncapped_h2": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2nocap-3h-H2.nc"),
    # --- With H2 + NH3 (full ammonia optionality) ---
    "co2_zero_nh3": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2zero-3h-NH3.nc"),
    # --- With H2 + NH3, DEA 2030 costs (2020 EUR) ---
    "co2_limited_nh3_dea30": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2L-3h-NH3-DEA30.nc"),
    "co2_zero_nh3_dea30": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2zero-3h-NH3-DEA30.nc"),
    "co2_uncapped_nh3_dea30": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2nocap-3h-NH3-DEA30.nc"),
    # --- DEA 2030 costs, explicit CO2 price 60 EUR/tCO2 (2020 EUR), no mass cap ---
    "co2_price_dea30": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2nocap-Ep60-3h-DEA30.nc"),
    "co2_price_h2_dea30": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2nocap-Ep60-3h-H2-DEA30.nc"),
    "co2_price_nh3_dea30": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2nocap-Ep60-3h-NH3-DEA30.nc"),
    # --- 1h temporal resolution sensitivity ---
    "co2_zero_nh3_dea30_1h": Path("../results/europe-year-140/networks/elec_s_140_ec_lcopt_Co2zero-1h-NH3-DEA30.nc"),
}

RESULT_CHOICE = "co2_zero_nh3_dea30"  # co2_limited | co2_zero | co2_uncapped | co2_limited_h2 | co2_zero_h2 | co2_uncapped_h2 | co2_zero_nh3 | co2_zero_nh3_dea30 | co2_limited_nh3_dea30 | co2_uncapped_nh3_dea30 | co2_price_dea30 | co2_price_h2_dea30 | co2_price_nh3_dea30 | co2_zero_nh3_dea30_1h
DEFAULT_RESULT_PATH = RUN_CANDIDATES.get(RESULT_CHOICE, RUN_CANDIDATES["co2_uncapped"])

candidate = RESULT_PATH_OVERRIDE if RESULT_PATH_OVERRIDE is not None else DEFAULT_RESULT_PATH
RESULT_PATH = candidate.resolve()

if not RESULT_PATH.exists():
    available = "\n".join(f"  - {k}: {v}" for k, v in RUN_CANDIDATES.items())
    raise FileNotFoundError(
        "Solved network missing at expected deterministic path:\n"
        f"  {RESULT_PATH}\n\n"
        "Either sync the file from ARC to this location, set RESULT_PATH_OVERRIDE, "
        "or switch RESULT_CHOICE.\n\nCandidates:\n"
        f"{available}"
    )

print("Using network file:", RESULT_PATH)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pypsa
except ImportError as exc:
    raise SystemExit('PyPSA missing. Install via mamba install -c conda-forge pypsa') from exc

try:
    import cartopy  # noqa: F401
    CARTOPY_AVAILABLE = True
except Exception:
    CARTOPY_AVAILABLE = False

print('PyPSA version:', pypsa.__version__)
print('Cartopy available:', CARTOPY_AVAILABLE)

In [ ]:
from pathlib import Path

# Compare metrics across the configured annual runs without changing RESULT_CHOICE.
run_candidates = RUN_CANDIDATES if "RUN_CANDIDATES" in globals() else {}


def _extract_co2_limit_tco2(n_local):
    gc = getattr(n_local, "global_constraints", None)
    if gc is None or gc.empty:
        return float("nan")

    mask = pd.Series(True, index=gc.index)
    if "type" in gc.columns:
        mask &= gc["type"].eq("primary_energy")
    if "carrier_attribute" in gc.columns:
        mask &= gc["carrier_attribute"].eq("co2_emissions")

    co2_rows = gc.loc[mask]
    if co2_rows.empty:
        co2_rows = gc.loc[gc.index.astype(str).str.contains("CO2", case=False, regex=True)]
        if co2_rows.empty:
            return float("nan")

    return float(co2_rows.iloc[0].get("constant", float("nan")))


def _compute_metrics(path):
    n_local = pypsa.Network(str(path))
    snapshot_weights = n_local.snapshot_weightings.objective.reindex(n_local.snapshots).fillna(1.0)
    total_demand_mwh = n_local.loads_t.p_set.mul(snapshot_weights, axis=0).sum().sum()
    objective_value = float(getattr(n_local, "objective", float("nan")))
    levelised_cost = (
        objective_value / total_demand_mwh if total_demand_mwh else float("nan")
    )

    co2_limit_tco2 = _extract_co2_limit_tco2(n_local)
    co2_limit_mt = co2_limit_tco2 / 1e6 if pd.notna(co2_limit_tco2) else float("nan")

    carrier_emissions = n_local.carriers.get("co2_emissions")
    total_co2_mt = float("nan")
    if carrier_emissions is not None and not carrier_emissions.isnull().all():

        def _component_emissions(power_df, component_df):
            if power_df is None or power_df.empty:
                return 0.0
            factors = component_df["carrier"].map(carrier_emissions).fillna(0.0)
            weighted = power_df.mul(snapshot_weights, axis=0)
            return weighted.mul(factors, axis=1).sum().sum()

        gen_emissions = _component_emissions(n_local.generators_t.p, n_local.generators)
        link_emissions = _component_emissions(getattr(n_local.links_t, "p0", None), n_local.links)
        store_emissions = _component_emissions(getattr(n_local.stores_t, "p", None), n_local.stores)
        total_co2_mt = (gen_emissions + link_emissions + store_emissions) / 1e6

    return pd.Series({
        "co2_limit_Mt": co2_limit_mt,
        "levelised_cost_EUR_per_MWh": levelised_cost,
        "total_operational_CO2_Mt": total_co2_mt,
    })


rows = {}
for key, path in run_candidates.items():
    candidate_path = Path(path).resolve()
    if not candidate_path.exists():
        rows[key] = pd.Series({
            "co2_limit_Mt": float("nan"),
            "levelised_cost_EUR_per_MWh": float("nan"),
            "total_operational_CO2_Mt": float("nan"),
        })
    else:
        rows[key] = _compute_metrics(candidate_path)
    rows[key]["path"] = str(candidate_path)

comparison = pd.DataFrame(rows).T
display(comparison)

In [ ]:
n = pypsa.Network(str(RESULT_PATH))
print(n)
print('Snapshots:', list(n.snapshots))
print('Buses:', len(n.buses), 'Lines:', len(n.lines), 'Generators:', len(n.generators))

## Objective (total system annuity) vs existing
- All monetary rows are in MEUR.
- Levelised cost is in EUR/MWh.

In [ ]:
def _series_or_zeros(df, col):
    if col in df.columns:
        return df[col].fillna(0.0)
    return pd.Series(0.0, index=df.index)


def _sum_existing_nameplate_times_capital_cost(df, base_col, cost_col):
    base = _series_or_zeros(df, base_col)
    cost = _series_or_zeros(df, cost_col)
    return (base * cost).sum()


objective_eur = float(n.objective) if hasattr(n, "objective") else float("nan")

sum_existing_nameplate_times_capital_cost_eur = (
    _sum_existing_nameplate_times_capital_cost(n.generators, "p_nom", "capital_cost")
    + _sum_existing_nameplate_times_capital_cost(n.storage_units, "p_nom", "capital_cost")
    + _sum_existing_nameplate_times_capital_cost(n.stores, "e_nom", "capital_cost")
    + _sum_existing_nameplate_times_capital_cost(n.lines, "s_nom", "capital_cost")
    + _sum_existing_nameplate_times_capital_cost(n.links, "p_nom", "capital_cost")
)

objective_minus_sum_existing_nameplate_times_capital_cost = (
    objective_eur - sum_existing_nameplate_times_capital_cost_eur if pd.notna(objective_eur) else float("nan")
)
objective_to_sum_existing_nameplate_times_capital_cost_ratio = (
    objective_eur / sum_existing_nameplate_times_capital_cost_eur
    if pd.notna(objective_eur) and sum_existing_nameplate_times_capital_cost_eur != 0
    else float("nan")
)

snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)
total_demand_mwh = n.loads_t.p_set.mul(snapshot_weights, axis=0).sum().sum()
levelised_cost_eur_per_mwh = (
    objective_eur / total_demand_mwh if pd.notna(objective_eur) and total_demand_mwh != 0 else float("nan")
)

carrier_emissions = n.carriers.get("co2_emissions")
if carrier_emissions is None or carrier_emissions.isnull().all():
    total_operational_co2_mt = float("nan")
else:
    def _component_emissions(power_df, component_df):
        if power_df is None or power_df.empty:
            return 0.0
        factors = component_df["carrier"].map(carrier_emissions).fillna(0.0)
        weighted = power_df.mul(snapshot_weights, axis=0)
        return weighted.mul(factors, axis=1).sum().sum()

    gen_emissions = _component_emissions(n.generators_t.p, n.generators)
    link_emissions = _component_emissions(getattr(n.links_t, "p0", None), n.links)
    store_emissions = _component_emissions(getattr(n.stores_t, "p", None), n.stores)
    total_operational_co2_mt = (gen_emissions + link_emissions + store_emissions) / 1e6

summary = pd.Series(
    {
        "objective_MEUR": objective_eur / 1e6,
        "sum_existing_nameplate_times_capital_cost_MEUR": sum_existing_nameplate_times_capital_cost_eur / 1e6,
        "objective_minus_sum_existing_nameplate_times_capital_cost_MEUR": objective_minus_sum_existing_nameplate_times_capital_cost / 1e6,
        "objective_to_sum_existing_nameplate_times_capital_cost_ratio": objective_to_sum_existing_nameplate_times_capital_cost_ratio,
        "levelised_cost_EUR_per_MWh": levelised_cost_eur_per_mwh,
        "total_operational_CO2_Mt": total_operational_co2_mt,
    }
)

display(summary.to_frame("value"))
print("Formula shown explicitly: sum(existing nameplate × capital_cost) over generators, storage_units, stores, lines, and links.")
print("Interpretation: objective includes operating costs and new investment; comparator is only existing nameplate × capital_cost.")
print("For comparison 2025 EU emissions 900 Mt (European Commission)")

## Installed capacities (MW)

In [ ]:
# Capacity plots: hide emergency load-shedding pseudo-generators by default
INCLUDE_LOAD_SHEDDING_IN_CAPACITY_PLOTS = False

# Generator capacities (p_nom_opt where available, else p_nom)
gen_cap_col = "p_nom_opt" if "p_nom_opt" in n.generators.columns else "p_nom"
cap_gen = n.generators.groupby("carrier")[gen_cap_col].sum()

# Link capacities (electrolysis, fuel cell, CCGT H2, pipelines, etc.)
link_cap_col = "p_nom_opt" if "p_nom_opt" in n.links.columns else "p_nom"
# Exclude DC transmission links — those are modeled as Lines
link_mask = n.links.carrier != "DC"
cap_links = n.links.loc[link_mask].groupby("carrier")[link_cap_col].sum()

# Store capacities (energy, MWh — shown separately)
store_cap_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"
cap_stores_mwh = n.stores.groupby("carrier")[store_cap_col].sum()

# Combine generator + link power capacities
cap_by_carrier_all = pd.concat([cap_gen, cap_links]).groupby(level=0).sum().sort_values(ascending=False)
if not INCLUDE_LOAD_SHEDDING_IN_CAPACITY_PLOTS:
    cap_by_carrier = cap_by_carrier_all.drop(index="load shedding", errors="ignore")
else:
    cap_by_carrier = cap_by_carrier_all.copy()

print("=== Power Capacities (MW) — Generators + Links ===")
cap_df = cap_by_carrier.to_frame(name="p_nom_opt_MW")
display(cap_df)

if not cap_stores_mwh.empty:
    print("\n=== Energy Storage Capacities (MWh) — Stores ===")
    display(cap_stores_mwh.to_frame(name="e_nom_opt_MWh"))

In [ ]:
title_suffix = " (excluding load shedding)" if not INCLUDE_LOAD_SHEDDING_IN_CAPACITY_PLOTS else ""
ax = cap_by_carrier.plot(kind="bar", figsize=(8, 4), title=f"Installed Capacities by Carrier{title_suffix}")
ax.set_ylabel("MW")
plt.tight_layout()
plt.show()

In [ ]:
# Why is load shedding capacity high but generation low?
print("- Load shedding is very expensive emergency supply (value of lost load).")
print("- Large p_nom is feasibility headroom; this does NOT imply large energy usage.")
print("- A typical run uses a tiny amount of shedding energy relative to total demand.")

## Generation mix (total over modeled period)

In [ ]:
# Aggregate generation/dispatch over the whole modeled period (MWh)
snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

# Generator output
gen_by_generator_mwh = n.generators_t.p.clip(lower=0.0).mul(snapshot_weights, axis=0).sum(axis=0)
gen_mix = (
    pd.DataFrame({"mwh": gen_by_generator_mwh, "carrier": n.generators["carrier"]})
    .groupby("carrier")["mwh"]
    .sum()
)

# Link dispatch (p0 = power drawn from bus0, positive = active)
# Show output-side energy for links that inject to AC (fuel cell, CCGT H2)
link_mask = n.links.carrier != "DC"
link_p0 = n.links_t.p0.loc[:, n.links_t.p0.columns.isin(n.links.loc[link_mask].index)]
link_energy = link_p0.clip(lower=0.0).mul(snapshot_weights, axis=0).sum(axis=0)
link_mix = (
    pd.DataFrame({"mwh": link_energy, "carrier": n.links.loc[link_mask, "carrier"]})
    .groupby("carrier")["mwh"]
    .sum()
)

tech_mix_total = pd.concat([gen_mix, link_mix]).groupby(level=0).sum().sort_values(ascending=False).rename("generation_MWh")
tech_mix_total

In [ ]:
tech_mix_total.plot(kind='bar', figsize=(8, 4), title='Generation Mix (total over modeled period)')
plt.ylabel('MWh')
plt.tight_layout()
plt.show()

## Installed Capacity Map (MW, output basis)

**Note on load shedding**
- `load shedding` is a high-penalty emergency pseudo-generator (`marginal_cost = 100000 EUR/MWh`).
- It is excluded from **capacity** visuals by default for readability.
- It is still included in dispatch/energy diagnostics, so generation totals remain physically/accounting consistent.

In [ ]:
import numpy as np
import matplotlib.ticker as mticker
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

if CARTOPY_AVAILABLE and not n.generators.empty:
    import cartopy.crs as ccrs
    from matplotlib.patches import Wedge

    # Capacity pies: hide emergency load shedding by default (toggle below if needed)
    INCLUDE_LOAD_SHEDDING_IN_CAPACITY_MAP = False
    gen_for_cap = n.generators if INCLUDE_LOAD_SHEDDING_IN_CAPACITY_MAP else n.generators[n.generators["carrier"] != "load shedding"]

    # ── Generator capacities by bus ──
    gen_cap_col = "p_nom_opt" if "p_nom_opt" in gen_for_cap.columns else "p_nom"
    gen_cap = (
        gen_for_cap.assign(capacity_MW=gen_for_cap[gen_cap_col].fillna(gen_for_cap["p_nom"]))
        .groupby(["bus", "carrier"])["capacity_MW"]
        .sum()
        .unstack(fill_value=0.0)
    )

    # ── Link capacities by AC bus — all non-DC links on OUTPUT basis ──
    # p_nom is input-side; multiply by efficiency for output-side MW
    ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])
    link_cap_col = "p_nom_opt" if "p_nom_opt" in n.links.columns else "p_nom"
    non_dc_links = n.links[n.links.carrier != "DC"].copy()
    # Map each link to its AC bus (bus0 or bus1, whichever is AC)
    non_dc_links["ac_bus"] = non_dc_links.apply(
        lambda r: r["bus0"] if r["bus0"] in ac_bus_set else (r["bus1"] if r["bus1"] in ac_bus_set else None),
        axis=1,
    )
    ac_links = non_dc_links.dropna(subset=["ac_bus"])
    if not ac_links.empty:
        # Output-side capacity: p_nom_opt × efficiency
        ac_links = ac_links.copy()
        ac_links["cap_out_MW"] = ac_links[link_cap_col] * ac_links["efficiency"].fillna(1.0)
        link_cap = (
            ac_links.groupby(["ac_bus", "carrier"])["cap_out_MW"]
            .sum()
            .unstack(fill_value=0.0)
        )
        link_cap.index.name = "bus"
        cap_df = pd.concat([gen_cap, link_cap]).groupby(level=0).sum().fillna(0.0)
    else:
        cap_df = gen_cap

    carriers = cap_df.columns.tolist()
    colors = plt.cm.tab20(np.linspace(0, 1, len(carriers)))
    color_map = dict(zip(carriers, colors))

    line_widths = 1e-4 * n.lines.s_nom_opt.fillna(n.lines.s_nom)
    link_widths = 1e-4 * n.links.p_nom_opt.fillna(n.links.p_nom)

    pies = cap_df[cap_df.sum(axis=1) > 0]
    if len(pies) == 0:
        print("No capacity data to plot.")
    else:
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})
        try:
            n.plot(
                ax=ax,
                boundaries=(-11, 30, 34, 72),
                bus_sizes=0.0,
                line_widths=line_widths,
                line_colors="gainsboro",
                link_widths=link_widths,
                link_colors="lightsteelblue",
                branch_components={"Line", "Link"},
            )
        except Exception as exc:
            print("n.plot failed; falling back to plain coastlines:", exc)
            ax.set_extent((-11, 30, 34, 72))
            ax.coastlines()

        max_total = pies.sum(axis=1).max()
        r_scale = 2.0 / max_total if max_total > 0 else 0.0

        for bus, row in pies.iterrows():
            if bus not in n.buses.index:
                continue
            x = n.buses.loc[bus, "x"]
            y = n.buses.loc[bus, "y"]
            total = row.sum()
            if total <= 0:
                continue
            start = 0.0
            radius = total * r_scale
            for carrier in carriers:
                val = row.get(carrier, 0.0)
                if val <= 0:
                    continue
                angle = 360 * val / total
                wedge = Wedge((x, y), radius, start, start + angle, facecolor=color_map[carrier], edgecolor="white", linewidth=0.3)
                ax.add_patch(wedge)
                start += angle

        legend_handles = [
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[c], label=c, markersize=8)
            for c in carriers
        ]
        ax.legend(handles=legend_handles, loc="upper left", fontsize="small", ncol=2, frameon=True)

        ax.set_axis_on()
        for spine in ax.spines.values():
            spine.set_visible(True)
        ax.set_xlim(-11, 30)
        ax.set_ylim(34, 72)

        major_xticks = np.arange(-10, 31, 5)
        major_yticks = np.arange(35, 73, 5)
        ax.set_xticks(major_xticks, crs=ccrs.PlateCarree())
        ax.set_yticks(major_yticks, crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.set_title(f"Installed Capacity by Bus — output basis (MW){title_suffix}")
elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping capacity pie map.")
else:
    print("No generators available in this network file.")

## Electricity Dispatch Mix (MWh)

In [ ]:
import numpy as np
import matplotlib.ticker as mticker
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

if CARTOPY_AVAILABLE and not n.generators.empty:
    import cartopy.crs as ccrs
    from matplotlib.patches import Wedge

    snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

    # ── Generator dispatch by bus ──
    gen_by_generator_mwh = n.generators_t.p.clip(lower=0.0).mul(snapshot_weights, axis=0).sum(axis=0)
    gen_df = (
        pd.DataFrame(
            {
                "generation_MWh": gen_by_generator_mwh,
                "bus": n.generators["bus"],
                "carrier": n.generators["carrier"],
            }
        )
        .groupby(["bus", "carrier"])["generation_MWh"]
        .sum()
        .unstack(fill_value=0.0)
    )

    # ── Link dispatch: only electricity-producing links (bus1 → AC bus) ──
    # Shows electricity output (p0 × efficiency) for CCGT H2, CCGT NH3, battery discharger, etc.
    ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])
    non_dc_links = n.links[n.links.carrier != "DC"].copy()
    elec_producing = non_dc_links[non_dc_links["bus1"].isin(ac_bus_set)].copy()
    elec_producing["ac_bus"] = elec_producing["bus1"]
    ac_links = elec_producing

    if not ac_links.empty and hasattr(n, "links_t") and not n.links_t.p0.empty:
        link_dispatch = n.links_t.p0.loc[:, n.links_t.p0.columns.isin(ac_links.index)]
        # Convert to electricity output: p0 × efficiency
        link_eff = ac_links.loc[link_dispatch.columns, "efficiency"].fillna(1.0)
        link_energy_mwh = (link_dispatch.clip(lower=0.0) * link_eff).mul(snapshot_weights, axis=0).sum(axis=0)
        link_gen_df = (
            pd.DataFrame({
                "generation_MWh": link_energy_mwh,
                "ac_bus": ac_links.loc[link_energy_mwh.index, "ac_bus"],
                "carrier": ac_links.loc[link_energy_mwh.index, "carrier"],
            })
            .groupby(["ac_bus", "carrier"])["generation_MWh"]
            .sum()
            .unstack(fill_value=0.0)
        )
        link_gen_df.index.name = "bus"
        combined_gen = pd.concat([gen_df, link_gen_df]).groupby(level=0).sum().fillna(0.0)
    else:
        combined_gen = gen_df

    carriers = combined_gen.columns.tolist()
    colors = plt.cm.tab20(np.linspace(0, 1, len(carriers)))
    color_map = dict(zip(carriers, colors))

    line_widths = 1e-4 * n.lines.s_nom_opt.fillna(n.lines.s_nom)
    link_widths = 1e-4 * n.links.p_nom_opt.fillna(n.links.p_nom)

    pies = combined_gen[combined_gen.sum(axis=1) > 0]
    if len(pies) == 0:
        print("No generation data to plot.")
    else:
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})
        try:
            n.plot(
                ax=ax,
                boundaries=(-11, 30, 34, 72),
                bus_sizes=0.0,
                line_widths=line_widths,
                line_colors="gainsboro",
                link_widths=link_widths,
                link_colors="lightsteelblue",
                branch_components={"Line", "Link"},
            )
        except Exception as exc:
            print("n.plot failed; falling back to plain coastlines:", exc)
            ax.set_extent((-11, 30, 34, 72))
            ax.coastlines()

        max_total = pies.sum(axis=1).max()
        r_scale = 2.0 / max_total if max_total > 0 else 0.0

        for bus, row in pies.iterrows():
            if bus not in n.buses.index:
                continue
            x = n.buses.loc[bus, "x"]
            y = n.buses.loc[bus, "y"]
            total = row.sum()
            if total <= 0:
                continue
            start = 0.0
            radius = total * r_scale
            for carrier in carriers:
                val = row.get(carrier, 0.0)
                if val <= 0:
                    continue
                angle = 360 * val / total
                wedge = Wedge((x, y), radius, start, start + angle, facecolor=color_map[carrier], edgecolor="white", linewidth=0.3)
                ax.add_patch(wedge)
                start += angle

        legend_handles = [
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[c], label=c, markersize=8)
            for c in carriers
        ]
        ax.legend(handles=legend_handles, loc="upper left", fontsize="small", ncol=2, frameon=True)

        ax.set_axis_on()
        for spine in ax.spines.values():
            spine.set_visible(True)
        ax.set_xlim(-11, 30)
        ax.set_ylim(34, 72)

        major_xticks = np.arange(-10, 31, 5)
        major_yticks = np.arange(35, 73, 5)
        ax.set_xticks(major_xticks, crs=ccrs.PlateCarree())
        ax.set_yticks(major_yticks, crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(axis="both", which="major", labelsize=8, length=6, width=0.6, direction="out")
        ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
        ax.yaxis.set_minor_locator(mticker.MultipleLocator(1))
        ax.set_title("Electricity Dispatch Mix (MWh)")
        plt.show()
elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping generation pie map.")
else:
    print("No generators available in this network file.")

## Storage Capacity Map (MWh)

Bubble map of **energy storage capacity** (`e_nom_opt`) by bus for each store carrier
(H2, NH3, battery).  Bubble area is proportional to capacity in MWh.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if CARTOPY_AVAILABLE and not n.stores.empty:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

    store_cap_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"

    # Group store capacity by bus and carrier
    store_cap = (
        n.stores.groupby(["bus", "carrier"])[store_cap_col]
        .sum()
        .unstack(fill_value=0.0)
    )

    # Map store buses to their coordinates (stores sit on H2/NH3/battery buses;
    # fall back to the parent AC bus coordinates via bus coordinates)
    store_buses = store_cap.index
    bus_x = n.buses.loc[store_buses, "x"]
    bus_y = n.buses.loc[store_buses, "y"]

    # Carrier styling
    store_carriers = store_cap.columns.tolist()
    store_colors = {
        "H2": "royalblue",
        "NH3": "seagreen",
        "battery": "goldenrod",
    }
    for c in store_carriers:
        if c not in store_colors:
            store_colors[c] = "mediumpurple"

    # Filter to buses with nonzero storage
    nonzero = store_cap[store_cap.sum(axis=1) > 0]

    if nonzero.empty:
        print("No storage capacity to plot.")
    else:
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

        # Background network
        line_widths = 1e-4 * n.lines.s_nom_opt.fillna(n.lines.s_nom)
        link_widths = 1e-4 * n.links.p_nom_opt.fillna(n.links.p_nom)
        try:
            n.plot(
                ax=ax, boundaries=(-11, 30, 34, 72), bus_sizes=0.0,
                line_widths=line_widths, line_colors="gainsboro",
                link_widths=link_widths, link_colors="lightsteelblue",
                branch_components={"Line", "Link"},
            )
        except Exception as exc:
            print("n.plot failed; falling back to coastlines:", exc)
            ax.set_extent((-11, 30, 34, 72))
            ax.coastlines()

        max_cap = nonzero.sum(axis=1).max()
        r_scale = 2.5 / max_cap if max_cap > 0 else 0.0

        from matplotlib.patches import Wedge

        for bus, row in nonzero.iterrows():
            if bus not in n.buses.index:
                continue
            x, y = n.buses.loc[bus, "x"], n.buses.loc[bus, "y"]
            total = row.sum()
            if total <= 0:
                continue
            radius = total * r_scale
            start = 0.0
            for carrier in store_carriers:
                val = row.get(carrier, 0.0)
                if val <= 0:
                    continue
                angle = 360 * val / total
                wedge = Wedge((x, y), radius, start, start + angle,
                              facecolor=store_colors.get(carrier, "grey"),
                              edgecolor="white", linewidth=0.3)
                ax.add_patch(wedge)
                start += angle

        # Legend
        legend_handles = [
            plt.Line2D([0], [0], marker="o", color="w",
                       markerfacecolor=store_colors.get(c, "grey"), label=c, markersize=8)
            for c in store_carriers if nonzero[c].sum() > 0
        ]
        ax.legend(handles=legend_handles, loc="upper left", fontsize="small", frameon=True)

        # Axes
        ax.set_axis_on()
        for spine in ax.spines.values():
            spine.set_visible(True)
        ax.set_xlim(-11, 30)
        ax.set_ylim(34, 72)
        major_xticks = np.arange(-10, 31, 5)
        major_yticks = np.arange(35, 73, 5)
        ax.set_xticks(major_xticks, crs=ccrs.PlateCarree())
        ax.set_yticks(major_yticks, crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(axis="both", which="major", labelsize=8, length=6, width=0.6, direction="out")
        ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
        ax.yaxis.set_minor_locator(mticker.MultipleLocator(1))
        ax.tick_params(axis="both", which="minor", length=3, width=0.4, direction="out")
        ax.set_title("Storage Capacity by Bus (MWh)")
        plt.tight_layout()
        plt.show()

        # Summary table
        print("\n=== Storage Capacity Summary (GWh) ===")
        store_gwh = nonzero.sum() / 1e3
        display(store_gwh.to_frame(name="total_GWh").style.format("{:,.1f}"))
elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping storage capacity map.")
else:
    print("No stores in this network.")

## Conversion Capacity Map (MW)

Pie-chart map of **conversion link capacity** (`p_nom_opt`) by bus for
non-transmission link carriers: electrolysis, NH3 synthesis, CCGT H2, CCGT NH3, etc.
Each link is mapped to its AC bus.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if CARTOPY_AVAILABLE and not n.links.empty:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    from matplotlib.patches import Wedge

    # Conversion carriers = everything except DC and pipeline transmission
    TRANSMISSION_CARRIERS = {"DC", "H2 pipeline", "NH3 pipeline"}
    conv_links = n.links[~n.links.carrier.isin(TRANSMISSION_CARRIERS)].copy()

    if conv_links.empty:
        print("No conversion links in this network.")
    else:
        # Map each conversion link to its AC bus
        ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])
        link_cap_col = "p_nom_opt" if "p_nom_opt" in conv_links.columns else "p_nom"
        conv_links["ac_bus"] = conv_links.apply(
            lambda r: r["bus0"] if r["bus0"] in ac_bus_set else (r["bus1"] if r["bus1"] in ac_bus_set else None),
            axis=1,
        )
        ac_conv = conv_links.dropna(subset=["ac_bus"])

        if ac_conv.empty:
            print("No conversion links connected to AC buses.")
        else:
            conv_cap = (
                ac_conv.groupby(["ac_bus", "carrier"])[link_cap_col]
                .sum()
                .unstack(fill_value=0.0)
            )
            conv_cap.index.name = "bus"

            conv_carriers = conv_cap.columns.tolist()
            conv_colors = {
                "H2 electrolysis": "royalblue",
                "CCGT H2": "steelblue",
                "NH3 synthesis": "seagreen",
                "CCGT NH3": "mediumseagreen",
                "battery charger": "goldenrod",
                "battery discharger": "orange",
            }
            _fallback_cmap = plt.cm.Set2(np.linspace(0, 1, max(len(conv_carriers), 1)))
            for i, c in enumerate(conv_carriers):
                if c not in conv_colors:
                    conv_colors[c] = _fallback_cmap[i % len(_fallback_cmap)]

            nonzero = conv_cap[conv_cap.sum(axis=1) > 0]

            if nonzero.empty:
                print("No nonzero conversion capacity to plot.")
            else:
                fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

                line_widths = 1e-4 * n.lines.s_nom_opt.fillna(n.lines.s_nom)
                link_widths = 1e-4 * n.links.p_nom_opt.fillna(n.links.p_nom)
                try:
                    n.plot(
                        ax=ax, boundaries=(-11, 30, 34, 72), bus_sizes=0.0,
                        line_widths=line_widths, line_colors="gainsboro",
                        link_widths=link_widths, link_colors="lightsteelblue",
                        branch_components={"Line", "Link"},
                    )
                except Exception as exc:
                    print("n.plot failed; falling back to coastlines:", exc)
                    ax.set_extent((-11, 30, 34, 72))
                    ax.coastlines()

                max_cap = nonzero.sum(axis=1).max()
                r_scale = 2.0 / max_cap if max_cap > 0 else 0.0

                for bus, row in nonzero.iterrows():
                    if bus not in n.buses.index:
                        continue
                    x, y = n.buses.loc[bus, "x"], n.buses.loc[bus, "y"]
                    total = row.sum()
                    if total <= 0:
                        continue
                    radius = total * r_scale
                    start = 0.0
                    for carrier in conv_carriers:
                        val = row.get(carrier, 0.0)
                        if val <= 0:
                            continue
                        angle = 360 * val / total
                        wedge = Wedge((x, y), radius, start, start + angle,
                                      facecolor=conv_colors.get(carrier, "grey"),
                                      edgecolor="white", linewidth=0.3)
                        ax.add_patch(wedge)
                        start += angle

                legend_handles = [
                    plt.Line2D([0], [0], marker="o", color="w",
                               markerfacecolor=conv_colors.get(c, "grey"), label=c, markersize=8)
                    for c in conv_carriers if nonzero[c].sum() > 0
                ]
                ax.legend(handles=legend_handles, loc="upper left", fontsize="small", frameon=True)

                ax.set_axis_on()
                for spine in ax.spines.values():
                    spine.set_visible(True)
                ax.set_xlim(-11, 30)
                ax.set_ylim(34, 72)
                major_xticks = np.arange(-10, 31, 5)
                major_yticks = np.arange(35, 73, 5)
                ax.set_xticks(major_xticks, crs=ccrs.PlateCarree())
                ax.set_yticks(major_yticks, crs=ccrs.PlateCarree())
                ax.xaxis.set_major_formatter(LongitudeFormatter())
                ax.yaxis.set_major_formatter(LatitudeFormatter())
                ax.tick_params(axis="both", which="major", labelsize=8, length=6, width=0.6, direction="out")
                ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
                ax.yaxis.set_minor_locator(mticker.MultipleLocator(1))
                ax.tick_params(axis="both", which="minor", length=3, width=0.4, direction="out")
                ax.set_title("Conversion Link Capacity by Bus (MW)")
                plt.tight_layout()
                plt.show()

                # Summary table
                print("\n=== Conversion Capacity Summary (GW) ===")
                conv_gw = nonzero.sum() / 1e3
                display(conv_gw.to_frame(name="total_GW").style.format("{:,.2f}"))
elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping conversion capacity map.")
else:
    print("No links in this network.")

## Conversion Throughput Map (TWh)

Pie-chart map of **annual energy throughput** (weighted `p0`, input-side) through
conversion links by AC bus.  Shows how heavily each conversion step is actually
used, complementing the capacity map above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if CARTOPY_AVAILABLE and not n.links.empty:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    from matplotlib.patches import Wedge

    snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

    TRANSMISSION_CARRIERS = {"DC", "H2 pipeline", "NH3 pipeline"}
    conv_links = n.links[~n.links.carrier.isin(TRANSMISSION_CARRIERS)].copy()

    if conv_links.empty:
        print("No conversion links in this network.")
    else:
        ac_bus_set = set(n.buses.index[n.buses.carrier == "AC"])
        conv_links["ac_bus"] = conv_links.apply(
            lambda r: r["bus0"] if r["bus0"] in ac_bus_set else (r["bus1"] if r["bus1"] in ac_bus_set else None),
            axis=1,
        )
        ac_conv = conv_links.dropna(subset=["ac_bus"])

        if ac_conv.empty:
            print("No conversion links connected to AC buses.")
        else:
            # Compute weighted throughput (input-side, TWh)
            p0_cols = [c for c in ac_conv.index if c in n.links_t.p0.columns]
            if not p0_cols:
                print("No dispatch data for conversion links.")
            else:
                throughput_mwh = (
                    n.links_t.p0[p0_cols]
                    .clip(lower=0.0)
                    .mul(snapshot_weights, axis=0)
                    .sum(axis=0)
                )
                ac_conv_disp = ac_conv.loc[p0_cols].copy()
                ac_conv_disp["throughput_twh"] = throughput_mwh.values / 1e6

                conv_tp = (
                    ac_conv_disp.groupby(["ac_bus", "carrier"])["throughput_twh"]
                    .sum()
                    .unstack(fill_value=0.0)
                )
                conv_tp.index.name = "bus"

                conv_carriers = conv_tp.columns.tolist()
                conv_colors = {
                    "H2 electrolysis": "royalblue",
                    "CCGT H2": "steelblue",
                    "NH3 synthesis": "seagreen",
                    "CCGT NH3": "mediumseagreen",
                    "battery charger": "goldenrod",
                    "battery discharger": "orange",
                }
                _fallback_cmap = plt.cm.Set2(np.linspace(0, 1, max(len(conv_carriers), 1)))
                for i, c in enumerate(conv_carriers):
                    if c not in conv_colors:
                        conv_colors[c] = _fallback_cmap[i % len(_fallback_cmap)]

                nonzero = conv_tp[conv_tp.sum(axis=1) > 0]

                if nonzero.empty:
                    print("No nonzero conversion throughput to plot.")
                else:
                    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

                    line_widths = 1e-4 * n.lines.s_nom_opt.fillna(n.lines.s_nom)
                    link_widths = 1e-4 * n.links.p_nom_opt.fillna(n.links.p_nom)
                    try:
                        n.plot(
                            ax=ax, boundaries=(-11, 30, 34, 72), bus_sizes=0.0,
                            line_widths=line_widths, line_colors="gainsboro",
                            link_widths=link_widths, link_colors="lightsteelblue",
                            branch_components={"Line", "Link"},
                        )
                    except Exception as exc:
                        print("n.plot failed; falling back to coastlines:", exc)
                        ax.set_extent((-11, 30, 34, 72))
                        ax.coastlines()

                    max_tp = nonzero.sum(axis=1).max()
                    r_scale = 2.0 / max_tp if max_tp > 0 else 0.0

                    for bus, row in nonzero.iterrows():
                        if bus not in n.buses.index:
                            continue
                        x, y = n.buses.loc[bus, "x"], n.buses.loc[bus, "y"]
                        total = row.sum()
                        if total <= 0:
                            continue
                        radius = total * r_scale
                        start = 0.0
                        for carrier in conv_carriers:
                            val = row.get(carrier, 0.0)
                            if val <= 0:
                                continue
                            angle = 360 * val / total
                            wedge = Wedge((x, y), radius, start, start + angle,
                                          facecolor=conv_colors.get(carrier, "grey"),
                                          edgecolor="white", linewidth=0.3)
                            ax.add_patch(wedge)
                            start += angle

                    legend_handles = [
                        plt.Line2D([0], [0], marker="o", color="w",
                                   markerfacecolor=conv_colors.get(c, "grey"), label=c, markersize=8)
                        for c in conv_carriers if nonzero[c].sum() > 0
                    ]
                    ax.legend(handles=legend_handles, loc="upper left", fontsize="small", frameon=True)

                    ax.set_axis_on()
                    for spine in ax.spines.values():
                        spine.set_visible(True)
                    ax.set_xlim(-11, 30)
                    ax.set_ylim(34, 72)
                    major_xticks = np.arange(-10, 31, 5)
                    major_yticks = np.arange(35, 73, 5)
                    ax.set_xticks(major_xticks, crs=ccrs.PlateCarree())
                    ax.set_yticks(major_yticks, crs=ccrs.PlateCarree())
                    ax.xaxis.set_major_formatter(LongitudeFormatter())
                    ax.yaxis.set_major_formatter(LatitudeFormatter())
                    ax.tick_params(axis="both", which="major", labelsize=8, length=6, width=0.6, direction="out")
                    ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
                    ax.yaxis.set_minor_locator(mticker.MultipleLocator(1))
                    ax.tick_params(axis="both", which="minor", length=3, width=0.4, direction="out")
                    ax.set_title("Conversion Link Throughput by Bus (TWh, input-side)")
                    plt.tight_layout()
                    plt.show()

                    # Summary table
                    print("\n=== Conversion Throughput Summary (TWh) ===")
                    conv_twh_total = nonzero.sum()
                    display(conv_twh_total.to_frame(name="total_TWh").style.format("{:,.1f}"))
elif not CARTOPY_AVAILABLE:
    print("Cartopy not available; skipping conversion throughput map.")
else:
    print("No links in this network.")

## Line loading distribution (all snapshots)

In [ ]:
import numpy as np

if "p0" in n.lines_t:
    loading_all = n.lines_t.p0.abs().div(n.lines.s_nom, axis=1)
    loading = loading_all.replace([np.inf, -np.inf], np.nan).stack().dropna()
    plt.figure(figsize=(6, 4))
    loading.clip(upper=1.5).hist(bins=40)
    plt.title("Line Loading Distribution (all snapshots)")
    plt.xlabel("Loading (p.u.)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Line flow results missing in network file.")

## Marginal prices (period summary)

In [ ]:
if "marginal_price" in n.buses_t:
    price_stats = pd.DataFrame({
        "mean_EUR_per_MWh": n.buses_t.marginal_price.mean(axis=0),
        "median_EUR_per_MWh": n.buses_t.marginal_price.median(axis=0),
        "p95_EUR_per_MWh": n.buses_t.marginal_price.quantile(0.95, axis=0),
    }).dropna().sort_values("mean_EUR_per_MWh", ascending=False)
    price_stats.head(10)
else:
    print("No marginal prices stored in buses_t.")

## Technology costs (EUR/MW installed)

In [ ]:
# ── Generator costs per MW of installed capacity ──
gen_cap_col = "p_nom_opt" if "p_nom_opt" in n.generators.columns else "p_nom"
gen_costs = n.generators.groupby("carrier").agg(
    total_capacity_MW=(gen_cap_col, "sum"),
    capital_cost_EUR_per_MW=("capital_cost", "mean"),
    marginal_cost_EUR_per_MWh=("marginal_cost", "mean"),
).sort_values("total_capacity_MW", ascending=False)
gen_costs = gen_costs[gen_costs.total_capacity_MW > 0]

print("=== Generator costs by carrier (EUR/MW annualised capital, EUR/MWh marginal) ===")
display(gen_costs.style.format({
    "total_capacity_MW": "{:,.0f}",
    "capital_cost_EUR_per_MW": "{:,.0f}",
    "marginal_cost_EUR_per_MWh": "{:.2f}",
}))

# ── Link costs per MW ──
link_cap_col = "p_nom_opt" if "p_nom_opt" in n.links.columns else "p_nom"
link_costs = n.links[n.links.carrier != "DC"].groupby("carrier").agg(
    total_capacity_MW=(link_cap_col, "sum"),
    capital_cost_EUR_per_MW=("capital_cost", "mean"),
    marginal_cost_EUR_per_MWh=("marginal_cost", "mean"),
).sort_values("total_capacity_MW", ascending=False)
link_costs = link_costs[link_costs.total_capacity_MW > 0]

if not link_costs.empty:
    print("\n=== Link costs by carrier (EUR/MW annualised capital, EUR/MWh marginal) ===")
    display(link_costs.style.format({
        "total_capacity_MW": "{:,.0f}",
        "capital_cost_EUR_per_MW": "{:,.0f}",
        "marginal_cost_EUR_per_MWh": "{:.2f}",
    }))

# ── Store costs per MWh ──
store_cap_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"
store_costs = n.stores.groupby("carrier").agg(
    total_capacity_MWh=(store_cap_col, "sum"),
    capital_cost_EUR_per_MWh=("capital_cost", "mean"),
    marginal_cost_EUR_per_MWh=("marginal_cost", "mean"),
).sort_values("total_capacity_MWh", ascending=False)
store_costs = store_costs[store_costs.total_capacity_MWh > 0]

if not store_costs.empty:
    print("\n=== Store costs by carrier (EUR/MWh annualised capital) ===")
    display(store_costs.style.format({
        "total_capacity_MWh": "{:,.0f}",
        "capital_cost_EUR_per_MWh": "{:,.2f}",
        "marginal_cost_EUR_per_MWh": "{:.4f}",
    }))

## Ammonia Investigation: Efficiency Chain & Storage Economics

Is the model's ammonia usage reasonable?  This section surfaces:
1. **Carrier conversion efficiencies** for every link carrier in the network
2. The full **roundtrip efficiency chain** electricity → H2 → NH3 → electricity
3. A **storage cost comparison** explaining why the optimiser prefers NH3 despite lower roundtrip efficiency

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1. Carrier conversion efficiencies — every link type in the network
# ═══════════════════════════════════════════════════════════════════════
link_eff = n.links.groupby("carrier").agg(
    count=("carrier", "size"),
    efficiency_mean=("efficiency", "mean"),
    efficiency_min=("efficiency", "min"),
    efficiency_max=("efficiency", "max"),
)
if "efficiency2" in n.links.columns:
    eff2_col = n.links.groupby("carrier")["efficiency2"].mean()
    link_eff["efficiency2_mean"] = eff2_col

print("=== Link Carrier Conversion Efficiencies ===")
display(link_eff.style.format(precision=4, na_rep="—"))

# ═══════════════════════════════════════════════════════════════════════
# 2. Roundtrip efficiency chains
# ═══════════════════════════════════════════════════════════════════════
eff = n.links.groupby("carrier")["efficiency"].mean()
eff2 = pd.Series(dtype=float)
if "efficiency2" in n.links.columns:
    eff2 = n.links.groupby("carrier")["efficiency2"].mean()

# Discover actual carrier names (case-insensitive matching)
carrier_names = {c.lower(): c for c in eff.index}
elec_name = carrier_names.get("h2 electrolysis")
fc_name = carrier_names.get("h2 fuel cell")
ccgt_h2_name = carrier_names.get("ccgt h2")
synth_name = carrier_names.get("nh3 synthesis")
ccgt_nh3_name = carrier_names.get("ccgt nh3")

chains = {}

# --- H2 roundtrip via fuel cell ---
if elec_name and fc_name:
    eta_elec = eff[elec_name]
    eta_fc = eff[fc_name]
    rt = eta_elec * eta_fc
    chains["el → H2 → el (fuel cell)"] = {
        "electrolysis": eta_elec,
        "reconversion": eta_fc,
        "roundtrip": rt,
    }

# --- H2 roundtrip via CCGT H2 ---
if elec_name and ccgt_h2_name:
    eta_elec = eff[elec_name]
    eta_ccgt_h2 = eff[ccgt_h2_name]
    rt = eta_elec * eta_ccgt_h2
    chains["el → H2 → el (CCGT H2)"] = {
        "electrolysis": eta_elec,
        "reconversion": eta_ccgt_h2,
        "roundtrip": rt,
    }

# --- NH3 roundtrip via CCGT NH3 ---
if elec_name and synth_name and ccgt_nh3_name:
    eta_elec = eff[elec_name]
    eta_synth = eff[synth_name]
    eta_ccgt_nh3 = eff[ccgt_nh3_name]
    elec_draw_per_h2 = abs(eff2.get(synth_name, 0.141))
    # For 1 MWh_el input to electrolysis:
    h2_out = eta_elec
    nh3_out = h2_out * eta_synth
    elec_consumed_hb = h2_out * elec_draw_per_h2
    total_el_in = 1.0 + elec_consumed_hb
    el_out = nh3_out * eta_ccgt_nh3
    rt = el_out / total_el_in
    chains["el → H2 → NH3 → el (CCGT NH3)"] = {
        "electrolysis": eta_elec,
        "NH3 synthesis (H2→NH3)": eta_synth,
        "Haber-Bosch el draw/MWh_H2": elec_draw_per_h2,
        "CCGT NH3 reconversion": eta_ccgt_nh3,
        "total_el_in_per_1MWh_electrolysis": total_el_in,
        "el_out": el_out,
        "roundtrip": rt,
    }

print("\n=== Roundtrip Efficiency Chains ===")
if not chains:
    print("  (No complete conversion chains found in this network)")
    print(f"  Available link carriers: {sorted(eff.index.tolist())}")
else:
    for name, data in chains.items():
        print(f"\n  {name}:")
        for k, v in data.items():
            print(f"    {k}: {v:.4f}" if isinstance(v, float) else f"    {k}: {v}")
        print(f"    → roundtrip efficiency: {data['roundtrip']:.1%}")

# ═══════════════════════════════════════════════════════════════════════
# 3. NH3 vs H2 storage: energy volumes & cost comparison
# ═══════════════════════════════════════════════════════════════════════
snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

link_throughput = {}
for carrier in n.links.carrier.unique():
    idx = n.links.index[n.links.carrier == carrier]
    if idx.empty or carrier == "DC":
        continue
    cols = [c for c in idx if c in n.links_t.p0.columns]
    if not cols:
        continue
    p0 = n.links_t.p0[cols].clip(lower=0.0).mul(snapshot_weights, axis=0).sum().sum()
    link_throughput[carrier] = p0

link_tp = pd.Series(link_throughput, name="throughput_MWh_input").sort_values(ascending=False)

store_cap_col = "e_nom_opt" if "e_nom_opt" in n.stores.columns else "e_nom"
store_summary = n.stores.groupby("carrier").agg(
    total_capacity_MWh=(store_cap_col, "sum"),
    capital_cost_EUR_per_MWh=("capital_cost", "mean"),
)

print("\n=== Link Throughput (MWh input, weighted) ===")
display(link_tp.to_frame())

print("\n=== Store Capacities & Costs ===")
display(store_summary.style.format({"total_capacity_MWh": "{:,.0f}", "capital_cost_EUR_per_MWh": "{:,.2f}"}))

# Sanity check: does synthesis output match CCGT consumption?
if synth_name and ccgt_nh3_name:
    synth_tp = link_tp.get(synth_name, 0)
    synth_nh3_out = synth_tp * eff.get(synth_name, 0.881)
    ccgt_tp = link_tp.get(ccgt_nh3_name, 0)
    print(f"\n=== NH3 Balance Check ===")
    print(f"  NH3 synthesis output: {synth_nh3_out/1e6:,.1f} TWh_NH3 (from {synth_tp/1e6:,.1f} TWh_H2 input)")
    print(f"  CCGT NH3 consumption: {ccgt_tp/1e6:,.1f} TWh_NH3")
    if ccgt_tp > 0 and synth_nh3_out > 0:
        ratio = ccgt_tp / synth_nh3_out
        if ratio > 2:
            print(f"  ⚠ CCGT NH3 consumes {ratio:.0f}× more NH3 than synthesis produces!")
            print(f"  This may indicate orphaned NH3 pipeline links or a model formulation issue.")

print("\n=== Cost insight: why the optimiser may prefer NH3 ===")
if "H2" in store_summary.index and "NH3" in store_summary.index:
    h2_store_cost = store_summary.loc["H2", "capital_cost_EUR_per_MWh"]
    nh3_store_cost = store_summary.loc["NH3", "capital_cost_EUR_per_MWh"]
    print(f"  H2 storage capital cost:  {h2_store_cost:,.2f} EUR/MWh_H2")
    print(f"  NH3 storage capital cost: {nh3_store_cost:,.2f} EUR/MWh_NH3")
    print(f"  NH3 is {h2_store_cost/nh3_store_cost:.0f}× cheaper per MWh of stored energy")
    print(f"  Even after accounting for worse roundtrip (~30% vs ~37%),")
    print(f"  the vastly cheaper bulk storage makes NH3 attractive for seasonal storage.")

## Levelised Cost of Green Ammonia (LCOA) — Import Break-Even

What does **domestically-produced green ammonia** actually cost the European system?
If a foreign supplier could deliver green NH3 at a lower price, the system would
prefer imports over building the full electrolysis → Haber-Bosch → storage chain.

**Two approaches:**

1. **Bottom-up LCOA** — Sum the annualised capital + operating costs of every
   component in the NH3 production chain (electrolysis, H2 buffer storage,
   NH3 synthesis, NH3 storage) and divide by total NH3 output.  This gives the
   full-chain cost in EUR/MWh_NH3 (and converted to USD/t_NH3 for comparison
   with global ammonia markets).

2. **Shadow price (dual)** — The time-weighted average marginal price at NH3
   buses from the LP dual variables.  This is the system's willingness-to-pay
   for one additional MWh of NH3 at the margin.

The bottom-up cost is the *average* cost; the shadow price is the *marginal* cost.
For a competitive import analysis, the bottom-up LCOA is more relevant — it tells
you the price below which imports would reduce total system cost.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Levelised Cost of Green Ammonia — Bottom-up & Shadow-price approaches
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np

snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

# ── 1. Identify NH3 chain components ─────────────────────────────────
carrier_map = {c.lower(): c for c in n.links.carrier.unique()}
synth_carrier = carrier_map.get("nh3 synthesis")
ccgt_nh3_carrier = carrier_map.get("ccgt nh3")
elec_carrier = carrier_map.get("h2 electrolysis")
pipeline_carrier = carrier_map.get("nh3 pipeline")

if not synth_carrier:
    print("No NH3 synthesis links found — this network has no ammonia chain.")
else:
    # ── Helper: annualised cost of a set of links ────────────────────
    def _link_annual_cost(carrier_name):
        """Total annualised cost (capital + VOM) for all links of a carrier."""
        sel = n.links[n.links.carrier == carrier_name]
        if sel.empty:
            return 0.0, 0.0, 0.0
        p_nom = sel["p_nom_opt"] if "p_nom_opt" in sel.columns else sel["p_nom"]
        cap = (sel["capital_cost"] * p_nom).sum()
        cols = [c for c in sel.index if c in n.links_t.p0.columns]
        if cols:
            dispatch_mwh = n.links_t.p0[cols].clip(lower=0).mul(snapshot_weights, axis=0).sum().sum()
        else:
            dispatch_mwh = 0.0
        vom = sel["marginal_cost"].mean() * dispatch_mwh
        return cap, vom, dispatch_mwh

    def _store_annual_cost(carrier_name):
        """Total annualised cost for all stores of a carrier."""
        sel = n.stores[n.stores.carrier == carrier_name]
        if sel.empty:
            return 0.0, 0.0
        e_nom = sel["e_nom_opt"] if "e_nom_opt" in sel.columns else sel["e_nom"]
        cap = (sel["capital_cost"] * e_nom).sum()
        total_mwh = e_nom.sum()
        return cap, total_mwh

    # ── 2. Compute costs for each chain element ─────────────────────
    elec_cap, elec_vom, elec_dispatch = _link_annual_cost(elec_carrier) if elec_carrier else (0, 0, 0)
    synth_cap, synth_vom, synth_dispatch = _link_annual_cost(synth_carrier)
    nh3_store_cap, nh3_store_mwh = _store_annual_cost("NH3")
    h2_store_cap, h2_store_mwh = _store_annual_cost("H2")
    pipe_cap, pipe_vom, pipe_dispatch = _link_annual_cost(pipeline_carrier) if pipeline_carrier else (0, 0, 0)

    # ── 3. Determine NH3 output ──────────────────────────────────────
    synth_links = n.links[n.links.carrier == synth_carrier]
    synth_eff = synth_links["efficiency"].mean()
    nh3_produced_mwh = synth_dispatch * synth_eff

    # ── 4. Attribute electrolysis & H2 storage to NH3 chain ──────────
    if elec_carrier:
        elec_links = n.links[n.links.carrier == elec_carrier]
        elec_cols = [c for c in elec_links.index if c in n.links_t.p0.columns]
        if elec_cols:
            total_h2_produced = (
                n.links_t.p0[elec_cols].clip(lower=0)
                .mul(snapshot_weights, axis=0).sum().sum()
                * elec_links["efficiency"].mean()
            )
        else:
            total_h2_produced = 0
        h2_to_nh3 = synth_dispatch
        nh3_share_of_h2 = h2_to_nh3 / total_h2_produced if total_h2_produced > 0 else 0
    else:
        nh3_share_of_h2 = 0
        total_h2_produced = 0

    elec_cap_nh3 = elec_cap * nh3_share_of_h2
    elec_vom_nh3 = elec_vom * nh3_share_of_h2
    h2_store_cap_nh3 = h2_store_cap * nh3_share_of_h2

    # ── 5. Electricity consumed & its cost ───────────────────────────
    elec_for_electrolysis = elec_dispatch * nh3_share_of_h2 if elec_carrier else 0
    if "efficiency2" in synth_links.columns:
        hb_elec_per_h2 = abs(synth_links["efficiency2"].mean())
    else:
        hb_elec_per_h2 = 0.141
    hb_elec_draw = synth_dispatch * hb_elec_per_h2
    total_elec_consumed = elec_for_electrolysis + hb_elec_draw

    # System-average electricity price from bus marginal prices
    # PyPSA marginal_price is the LP dual, already in EUR/MWh
    avg_elec_price = 0.0
    if hasattr(n, "buses_t") and "marginal_price" in n.buses_t:
        ac_buses_idx = n.buses.index[n.buses.carrier == "AC"]
        mp_cols = [c for c in ac_buses_idx if c in n.buses_t.marginal_price.columns]
        if mp_cols:
            prices = n.buses_t.marginal_price[mp_cols]
            # Check for extreme outliers (load shedding / slack bus scarcity)
            flat_prices = prices.values.flatten()
            flat_prices = flat_prices[~np.isnan(flat_prices)]

            # Use median to avoid scarcity-spike bias
            median_price = np.median(flat_prices)
            mean_price = flat_prices.mean()

            # Trim extreme percentiles for a robust average
            p5 = np.percentile(flat_prices, 5)
            p95 = np.percentile(flat_prices, 95)
            trimmed = flat_prices[(flat_prices >= p5) & (flat_prices <= p95)]
            trimmed_mean = trimmed.mean() if len(trimmed) > 0 else mean_price

            avg_elec_price = trimmed_mean

    elec_cost_nh3 = total_elec_consumed * avg_elec_price

    # ── 6. Bottom-up LCOA ────────────────────────────────────────────
    total_capital = elec_cap_nh3 + synth_cap + nh3_store_cap + h2_store_cap_nh3 + pipe_cap
    total_vom = elec_vom_nh3 + synth_vom + pipe_vom
    total_cost = total_capital + total_vom + elec_cost_nh3

    lcoa_eur_per_mwh = total_cost / nh3_produced_mwh if nh3_produced_mwh > 0 else float("nan")

    MWH_PER_TONNE_NH3 = 5.17  # LHV basis (18.6 GJ/t)
    EUR_TO_USD = 1.10
    lcoa_usd_per_tonne = lcoa_eur_per_mwh * MWH_PER_TONNE_NH3 * EUR_TO_USD

    # ── 7. Shadow price at NH3 buses ─────────────────────────────────
    avg_nh3_shadow = float("nan")
    shadow_p10 = shadow_p90 = float("nan")
    nh3_buses = n.buses.index[n.buses.carrier == "NH3"]
    if hasattr(n, "buses_t") and "marginal_price" in n.buses_t:
        mp_nh3_cols = [c for c in nh3_buses if c in n.buses_t.marginal_price.columns]
        if mp_nh3_cols:
            nh3_prices = n.buses_t.marginal_price[mp_nh3_cols]
            flat_nh3 = nh3_prices.values.flatten()
            flat_nh3 = flat_nh3[~np.isnan(flat_nh3)]
            avg_nh3_shadow = np.median(flat_nh3) if len(flat_nh3) > 0 else float("nan")
            shadow_p10 = np.percentile(flat_nh3, 10) if len(flat_nh3) > 0 else float("nan")
            shadow_p90 = np.percentile(flat_nh3, 90) if len(flat_nh3) > 0 else float("nan")

    shadow_usd_per_tonne = avg_nh3_shadow * MWH_PER_TONNE_NH3 * EUR_TO_USD

    # ── 8. Diagnostics ───────────────────────────────────────────────
    print("=" * 70)
    print("DIAGNOSTICS — marginal price sanity check")
    print("-" * 70)
    if mp_cols:
        print(f"  AC electricity marginal_price (EUR/MWh):")
        print(f"    mean:     {mean_price:>10,.1f}")
        print(f"    median:   {median_price:>10,.1f}")
        print(f"    P5:       {p5:>10,.1f}")
        print(f"    P95:      {p95:>10,.1f}")
        print(f"    min:      {flat_prices.min():>10,.1f}")
        print(f"    max:      {flat_prices.max():>10,.1f}")
        print(f"    trimmed mean (P5-P95): {trimmed_mean:>8,.1f} ← used for LCOA")
        extreme_count = ((flat_prices < p5) | (flat_prices > p95)).sum()
        print(f"    extreme snapshots trimmed: {extreme_count:,} of {len(flat_prices):,}")
    print(f"  snapshot_weights: min={snapshot_weights.min():.1f}, max={snapshot_weights.max():.1f}, sum={snapshot_weights.sum():.0f}")

    # ── 9. Results ───────────────────────────────────────────────────
    print(f"\n{'=' * 70}")
    print("LEVELISED COST OF GREEN AMMONIA — IMPORT BREAK-EVEN ANALYSIS")
    print("=" * 70)

    print(f"\n{'Component':<35} {'Annual Cost (M€)':>18} {'Share':>8}")
    print("-" * 65)
    components = [
        ("Electrolysis (NH3 share)", elec_cap_nh3 + elec_vom_nh3),
        ("H2 storage (NH3 share)", h2_store_cap_nh3),
        ("NH3 synthesis (Haber-Bosch)", synth_cap + synth_vom),
        ("NH3 storage", nh3_store_cap),
        ("NH3 pipeline", pipe_cap + pipe_vom),
        ("Input electricity", elec_cost_nh3),
    ]
    for name, cost in components:
        share = cost / total_cost * 100 if total_cost > 0 else 0
        print(f"  {name:<33} {cost/1e6:>14,.1f} M{chr(8364)} {share:>6.1f}%")
    print("-" * 65)
    print(f"  {'TOTAL':<33} {total_cost/1e6:>14,.1f} M{chr(8364)} {'100.0':>6s}%")

    print(f"\n  NH3 produced:  {nh3_produced_mwh/1e6:,.2f} TWh_NH3")
    print(f"  H2 share going to NH3: {nh3_share_of_h2:.1%}")
    print(f"  Total elec consumed: {total_elec_consumed/1e6:,.2f} TWh_el")
    print(f"  Avg system elec price (trimmed): {avg_elec_price:,.1f} EUR/MWh_el")

    print(f"\n{'=' * 65}")
    print(f"  BOTTOM-UP LCOA:    {lcoa_eur_per_mwh:>8,.1f} EUR/MWh_NH3")
    print(f"                     {lcoa_usd_per_tonne:>8,.0f} USD/t_NH3")
    print(f"\n  SHADOW PRICE:      {avg_nh3_shadow:>8,.1f} EUR/MWh_NH3  (median)")
    print(f"                     {shadow_usd_per_tonne:>8,.0f} USD/t_NH3")
    print(f"                     P10={shadow_p10:,.1f}  P90={shadow_p90:,.1f} EUR/MWh_NH3")

    print(f"\n{'=' * 65}")
    print(f"  If green NH3 can be imported below ~{lcoa_usd_per_tonne:,.0f} USD/t")
    print(f"  ({lcoa_eur_per_mwh:,.0f} EUR/MWh), the European system would benefit")
    print(f"  from imports over domestic production.")

    print(f"\n{'=' * 65}")
    print("CONTEXT: Global green ammonia price estimates (2030)")
    print("-" * 65)
    benchmarks = [
        ("IRENA 2022 best locations (MENA, Chile)", "320-480"),
        ("BloombergNEF 2024 Australia/MENA export", "400-600"),
        ("IEA NZE 2030 global average", "500-700"),
        ("EU domestic (this model)", f"{lcoa_usd_per_tonne:,.0f}"),
    ]
    for source, price_range in benchmarks:
        print(f"  {source:<50} {price_range:>12} USD/t")

    summary_data = {
        "Bottom-up LCOA (EUR/MWh_NH3)": lcoa_eur_per_mwh,
        "Bottom-up LCOA (USD/t_NH3)": lcoa_usd_per_tonne,
        "Shadow price median (EUR/MWh_NH3)": avg_nh3_shadow,
        "Shadow price median (USD/t_NH3)": shadow_usd_per_tonne,
        "NH3 produced (TWh)": nh3_produced_mwh / 1e6,
        "Electrolysis -> NH3 share": nh3_share_of_h2,
        "Avg elec price trimmed (EUR/MWh)": avg_elec_price,
    }
    display(pd.Series(summary_data, name="value").to_frame().style.format("{:,.2f}"))

## Net Energy Flow Map by Carrier

Directed flow map showing **where energy actually moves** across the network.
Each branch shows the net annual energy flow (TWh) with an arrowhead indicating
direction. Line width is proportional to flow magnitude. Carriers are drawn as
separate layers so overlapping corridors are visible.

- **AC lines** — yellow, background layer
- **DC links** — orange
- **H2 pipeline** — blue
- **NH3 pipeline** — green

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyArrowPatch

if CARTOPY_AVAILABLE:
    import cartopy.crs as ccrs
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

    snapshot_weights = n.snapshot_weightings.objective.reindex(n.snapshots).fillna(1.0)

    # ── Compute net annual energy flow for AC lines ──
    line_flows = []
    if "p0" in n.lines_t:
        for idx, row in n.lines.iterrows():
            b0, b1 = row["bus0"], row["bus1"]
            if b0 not in n.buses.index or b1 not in n.buses.index:
                continue
            if idx not in n.lines_t.p0.columns:
                continue
            net_mwh = (n.lines_t.p0[idx] * snapshot_weights).sum()  # positive = bus0→bus1
            line_flows.append({
                "bus0": b0, "bus1": b1,
                "net_twh": net_mwh / 1e6,
                "carrier": "AC",
            })
    line_flows = pd.DataFrame(line_flows) if line_flows else pd.DataFrame()

    # ── Compute net annual energy flow for Links (DC, H2 pipeline, NH3 pipeline) ──
    transmission_carriers = ["DC", "H2 pipeline", "NH3 pipeline"]
    # Include any other pipeline carriers
    for c in n.links.carrier.unique():
        if "pipeline" in c.lower() and c not in transmission_carriers:
            transmission_carriers.append(c)

    link_flows = []
    for idx, row in n.links.iterrows():
        if row["carrier"] not in transmission_carriers:
            continue
        b0, b1 = row["bus0"], row["bus1"]
        if b0 not in n.buses.index or b1 not in n.buses.index:
            continue
        if idx not in n.links_t.p0.columns:
            continue
        net_mwh = (n.links_t.p0[idx] * snapshot_weights).sum()
        link_flows.append({
            "bus0": b0, "bus1": b1,
            "net_twh": net_mwh / 1e6,
            "carrier": row["carrier"],
        })
    link_flows = pd.DataFrame(link_flows) if link_flows else pd.DataFrame()

    # Aggregate bidirectional link pairs (A→B and B→A reversed) per carrier
    if not link_flows.empty:
        # Create canonical bus pair key (sorted) to aggregate forward + reversed
        link_flows["pair"] = link_flows.apply(
            lambda r: tuple(sorted([r["bus0"], r["bus1"]])), axis=1
        )
        # For each pair+carrier: sum net flows, track direction
        agg = link_flows.groupby(["pair", "carrier"]).agg(
            net_twh=("net_twh", "sum"),
            bus0=("bus0", "first"),
            bus1=("bus1", "first"),
        ).reset_index()
        link_flows = agg[["bus0", "bus1", "net_twh", "carrier"]]

    # ── Carrier styling ──
    carrier_colors = {
        "AC": "#FFD700",
        "DC": "darkorange",
        "H2 pipeline": "royalblue",
        "NH3 pipeline": "seagreen",
    }
    carrier_zorder = {"AC": 1, "DC": 2, "H2 pipeline": 3, "NH3 pipeline": 4}
    carrier_alpha = {"AC": 0.7, "DC": 0.8, "H2 pipeline": 0.8, "NH3 pipeline": 0.8}

    # ── Draw ──
    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.coastlines(resolution="50m", linewidth=0.4, color="grey")
    ax.set_extent((-11, 30, 34, 72), crs=ccrs.PlateCarree())

    def draw_flow_arrows(df, carrier, max_twh_global, ax):
        """Draw flow lines with arrowheads for a single carrier."""
        if df.empty:
            return
        color = carrier_colors.get(carrier, "purple")
        zorder = carrier_zorder.get(carrier, 2)
        alpha = carrier_alpha.get(carrier, 0.7)
        max_width = 5.0 if carrier != "AC" else 3.5
        min_width = 0.2 if carrier != "AC" else 0.15

        for _, row in df.iterrows():
            net = row["net_twh"]
            if abs(net) < 0.001:  # skip negligible flows
                continue
            b0, b1 = row["bus0"], row["bus1"]
            x0, y0 = n.buses.loc[b0, "x"], n.buses.loc[b0, "y"]
            x1, y1 = n.buses.loc[b1, "x"], n.buses.loc[b1, "y"]

            # Direction: positive net = bus0→bus1
            if net < 0:
                x0, y0, x1, y1 = x1, y1, x0, y0
                net = -net

            w = min_width + (max_width - min_width) * (net / max_twh_global) ** 0.5

            # Draw line
            ax.plot([x0, x1], [y0, y1], color=color, linewidth=w,
                    solid_capstyle="round", transform=ccrs.PlateCarree(),
                    zorder=zorder, alpha=alpha)

            # Arrowhead at 60% along the line (subtle directional indicator)
            frac = 0.6
            mx = x0 + frac * (x1 - x0)
            my = y0 + frac * (y1 - y0)
            dx = (x1 - x0) * 0.001  # tiny nudge for arrow direction
            dy = (y1 - y0) * 0.001
            ax.annotate("", xy=(mx + dx, my + dy), xytext=(mx, my),
                        arrowprops=dict(arrowstyle="-|>", color=color, lw=max(w * 0.6, 0.8),
                                        mutation_scale=max(w * 3, 6)),
                        transform=ccrs.PlateCarree(), zorder=zorder + 0.5)

    # Find global max for consistent scaling
    all_abs = []
    if not line_flows.empty:
        all_abs.extend(line_flows.net_twh.abs().tolist())
    if not link_flows.empty:
        all_abs.extend(link_flows.net_twh.abs().tolist())
    max_twh = max(all_abs) if all_abs else 1.0

    # Draw carriers in order (AC first as background)
    if not line_flows.empty:
        draw_flow_arrows(line_flows, "AC", max_twh, ax)
    for carrier in transmission_carriers:
        if not link_flows.empty:
            carrier_df = link_flows[link_flows.carrier == carrier]
            draw_flow_arrows(carrier_df, carrier, max_twh, ax)

    # ── Bus dots ──
    ac_buses = n.buses[n.buses.carrier == "AC"]
    ax.scatter(ac_buses.x, ac_buses.y, s=10, color="black", zorder=10,
               transform=ccrs.PlateCarree())

    # ── Legend with total flows ──
    legend_items = []
    if not line_flows.empty:
        ac_total = line_flows.net_twh.abs().sum()
        legend_items.append(
            plt.Line2D([0], [0], color=carrier_colors["AC"], linewidth=4,
                       label=f"AC ({ac_total:,.0f} TWh)")
        )
    for carrier in transmission_carriers:
        color = carrier_colors.get(carrier, "purple")
        if not link_flows.empty:
            c_df = link_flows[link_flows.carrier == carrier]
            if not c_df.empty:
                total = c_df.net_twh.abs().sum()
                legend_items.append(
                    plt.Line2D([0], [0], color=color, linewidth=4,
                               label=f"{carrier} ({total:,.0f} TWh)")
                )
    if legend_items:
        ax.legend(handles=legend_items, loc="upper left", fontsize="small",
                  frameon=True, title="Net annual flow")

    # ── Axes formatting ──
    major_xticks = np.arange(-10, 31, 5)
    major_yticks = np.arange(35, 73, 5)
    ax.set_xticks(major_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(major_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(axis="both", which="major", labelsize=8, length=6, width=0.6, direction="out")
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(1))
    ax.tick_params(axis="both", which="minor", length=3, width=0.4, direction="out")
    ax.set_title("Net Annual Energy Flows by Carrier (arrow = dominant direction)")
    plt.tight_layout()
    plt.show()

    # ── Summary table ──
    print("\n=== Net Flow Summary (TWh, absolute) ===")
    summary = {}
    if not line_flows.empty:
        summary["AC"] = line_flows.net_twh.abs().sum()
    for carrier in transmission_carriers:
        if not link_flows.empty:
            c_df = link_flows[link_flows.carrier == carrier]
            if not c_df.empty:
                summary[carrier] = c_df.net_twh.abs().sum()
    if summary:
        display(pd.Series(summary, name="net_flow_TWh").to_frame().style.format("{:,.1f}"))
else:
    print("Cartopy not available; skipping flow map.")